# Modelling panel — EDA and preparation

Panel-level cleaning and prep on the supervised **pathogen-long** units from notebook `10` (B2/A3 contract).

| Item | Detail |
|------|--------|
| Input | `modelling_panel.parquet` + `modelling_panel_species.parquet` + `modelling_panel_manifest.json` |
| Grain | Observed VectorNet × pathogen: Tier 1 `(h3_cell, year, month, vector_group, pathogen_name)`; Tier 2 adds `country_iso2` |
| Primary target | `pathogen_within_5y` (forward ≤5y, same key × **this** pathogen) |
| Secondary / QA | `pathogen_within_10y`, ever / delay / same-month; source & geo shares for this pathogen |
| Features (vector) | Group aggregates + `species_richness`; priority `vector_count_<slug>` via **unit-grain pivot** then merge |
| Training filter (upstream) | VectorNet coverage required — virus-only units already excluded; no synthetic empty months |
| Spec | [`specs/03`](specs/03-spatial-temporal-matching-strategy.md) + [`specs/04`](specs/04-reference-data-and-panel-pipeline.md) |

**Prerequisite:** run [`notebooks/01_sources/10_merge_modelling_panel.ipynb`](notebooks/01_sources/10_merge_modelling_panel.ipynb) so the panel **and** species side table exist.

**Species features:** after panel filters, take distinct unit keys → restrict species-long → pivot closed priority taxa at **unit** grain → left-merge onto pathogen-long rows (never join species-long before pivot).

This notebook covers load → profile → missingness / outliers → geo/time filters → unit-grain species pivot → modelling handoff. Stratify EDA by `pathogen_name`. Distribution plots, target deep-dive, feature importance, and model training come in later cells.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

from vector_guard.data import default_processed_dir, find_repo_root
from vector_guard.panel_features import (
    SPECIES_LONG_COLS,
    UNIT_KEY_COLS,
    attach_priority_species_features,
    priority_count_columns,
    validate_species_long,
)
from vector_guard.resolve import H3_RESOLUTION, load_priority_vector_taxa

REPO_ROOT = find_repo_root(Path.cwd())
PROCESSED_DIR = default_processed_dir(REPO_ROOT)
PANEL_PATH = PROCESSED_DIR / "modelling_panel.parquet"
SPECIES_PATH = PROCESSED_DIR / "modelling_panel_species.parquet"
MANIFEST_PATH = PROCESSED_DIR / "modelling_panel_manifest.json"

# Modelling window: denser VectorNet era (inclusive).
YEAR_MIN = 2010
YEAR_MAX = None  # inclusive upper bound; None = no cap

# Winsorize heavy-tailed vector *effort* proxies only (not priority species counts).
COUNT_WINSOR_Q = 0.99
COUNT_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
]

IDENTITY_COLS = [
    "spatial_tier",
    "h3_cell",
    "country_iso2",
    "year",
    "month",
    "vector_group",
    "pathogen_name",
]
TIER1_KEY = ["h3_cell", "year", "month", "vector_group", "pathogen_name"]
TIER2_KEY = ["country_iso2", "year", "month", "vector_group", "pathogen_name"]

# Primary supervised target (spec 04).
TARGET_COL = "pathogen_within_5y"
LABEL_COLS = [
    "pathogen_within_5y",
    "pathogen_within_10y",
    "pathogen_ever_before",
    "pathogen_ever_after",
    "pathogen_same_month",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "years_to_first_pathogen_after",
    "years_since_last_pathogen_before",
]

# Locked priority taxa reference (closed set). Wide vector_count_* names and
# FEATURE_COLS are built *after* unit-grain pivot onto panel_prep — not assumed
# present on the raw merge panel.
_priority = load_priority_vector_taxa()
PRIORITY_COUNT_COLS: list[str] = []  # filled after attach_priority_species_features
FEATURE_COLS_BASE = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "species_richness",
    "pathogen_ever_before",  # history before row time; not the forward target
]
FEATURE_COLS: list[str] = list(FEATURE_COLS_BASE)

# Forward-window QA columns (not modelling features — would leak label horizon).
SHARE_COLS = [
    "share_pathogen_exact_coords",
    "share_pathogen_subnational_centroid",
]

{
    "REPO_ROOT": str(REPO_ROOT),
    "PANEL_PATH": str(PANEL_PATH.relative_to(REPO_ROOT)),
    "SPECIES_PATH": str(SPECIES_PATH.relative_to(REPO_ROOT)),
    "H3_RESOLUTION": H3_RESOLUTION,
    "YEAR_MIN": YEAR_MIN,
    "n_priority_taxa_ref": len(_priority),
    "winsorize_cols": COUNT_COLS,
}


{'REPO_ROOT': '/workspaces/vector-guard',
 'PANEL_PATH': 'data/03_processed/modelling_panel.parquet',
 'SPECIES_PATH': 'data/03_processed/modelling_panel_species.parquet',
 'H3_RESOLUTION': 4,
 'YEAR_MIN': 2010,
 'n_priority_taxa_ref': 52,
 'winsorize_cols': ['vector_detection_count',
  'individual_count_sum',
  'individual_count_mean']}

## Load panel, species side table, and merge manifest

Do **not** require `vector_count_*` on the raw panel. Priority species features are attached after filters via unit-grain pivot from `modelling_panel_species.parquet`.

In [2]:
if not PANEL_PATH.exists():
    raise FileNotFoundError(
        f"Missing {PANEL_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )
if not SPECIES_PATH.exists():
    raise FileNotFoundError(
        f"Missing {SPECIES_PATH}. Run notebooks/01_sources/10_merge_modelling_panel.ipynb first."
    )

panel = pd.read_parquet(PANEL_PATH)
species_long = validate_species_long(pd.read_parquet(SPECIES_PATH))
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8")) if MANIFEST_PATH.exists() else {}

assert "pathogen_name" in panel.columns
assert TARGET_COL in panel.columns
assert "virus_within_5y" not in panel.columns
assert "pathogen_name" not in species_long.columns
assert list(species_long.columns) == SPECIES_LONG_COLS
# Do not assert PRIORITY_COUNT_COLS on the raw panel — they are built after pivot.

{
    "rows": len(panel),
    "cols": len(panel.columns),
    "species_rows": len(species_long),
    "species_units": int(species_long.loc[:, UNIT_KEY_COLS].drop_duplicates().shape[0]),
    "manifest_panel_rows": manifest.get("panel_rows"),
    "manifest_species_table_rows": manifest.get("species_table_rows"),
    "manifest_date_range": manifest.get("date_range"),
    "primary_label": manifest.get("primary_label"),
    "secondary_label": manifest.get("secondary_label"),
    "eligibility": manifest.get("eligibility"),
    "n_pathogens": manifest.get("n_pathogens"),
    "pathogen_positive_units": manifest.get("pathogen_positive_units"),
    "pathogen_positive_rate": manifest.get("pathogen_positive_rate"),
    "pathogen_within_10y_units": manifest.get("pathogen_within_10y_units"),
    "pathogen_ever_after_units": manifest.get("pathogen_ever_after_units"),
    "pathogen_ever_before_units": manifest.get("pathogen_ever_before_units"),
    "pathogen_same_month_units": manifest.get("pathogen_same_month_units"),
    "rows_by_spatial_tier": manifest.get("rows_by_spatial_tier"),
    "training_filter": manifest.get("training_filter"),
}


{'rows': 112134,
 'cols': 27,
 'species_rows': 31108,
 'species_units': 12995,
 'manifest_panel_rows': 112134,
 'manifest_species_table_rows': 31108,
 'manifest_date_range': {'max': '2025-04', 'min': '1957-09'},
 'primary_label': 'pathogen_within_5y',
 'secondary_label': 'pathogen_within_10y',
 'eligibility': 'from_first_vector',
 'n_pathogens': 26,
 'pathogen_positive_units': 1740,
 'pathogen_positive_rate': 0.015517149125153834,
 'pathogen_within_10y_units': 3417,
 'pathogen_ever_after_units': 6102,
 'pathogen_ever_before_units': 1388,
 'pathogen_same_month_units': 46,
 'rows_by_spatial_tier': {'h3': 112134},
 'training_filter': 'Supervised panel includes only observed VectorNet months (B2 pathogen-long); virus-only units are excluded; no synthetic empty months after first vector (A3 clock only). Primary label pathogen_within_5y is forward from the row time in the same spatial key x pathogen_name (within vector_group). Zeros / null delays mean no reported detection of that pathogen i

## Panel profile

Shape, dtypes, grain uniqueness, and basic value ranges.

In [3]:
display(panel.dtypes.to_frame("dtype"))
display(panel.head(5))

tier1 = panel.loc[panel["spatial_tier"] == "h3"]
tier2 = panel.loc[panel["spatial_tier"] == "country"]

profile = {
    "n_rows": len(panel),
    "n_tier1_h3": len(tier1),
    "n_tier2_country": len(tier2),
    "dup_tier1_keys": int(tier1.duplicated(TIER1_KEY).sum()) if len(tier1) else 0,
    "dup_tier2_keys": int(tier2.duplicated(TIER2_KEY).sum()) if len(tier2) else 0,
    "year_min": int(panel["year"].min()),
    "year_max": int(panel["year"].max()),
    "n_h3_cells": int(panel["h3_cell"].nunique(dropna=True)),
    "n_countries": int(panel["country_iso2"].nunique(dropna=True)),
    "n_pathogens": int(panel["pathogen_name"].nunique()),
    "vector_groups": sorted(panel["vector_group"].dropna().unique().tolist()),
    "vector_present_all_one": bool((panel["vector_present"] == 1).all()),
    "pathogen_positive_5y": int(panel[TARGET_COL].sum()),
    "pathogen_positive_rate_5y": float(panel[TARGET_COL].mean()),
    "pathogen_within_10y": int(panel["pathogen_within_10y"].sum()),
    "pathogen_ever_after": int(panel["pathogen_ever_after"].sum()),
    "pathogen_ever_before": int(panel["pathogen_ever_before"].sum()),
    "pathogen_same_month": int(panel["pathogen_same_month"].sum()),
    "species_richness_mean": float(panel["species_richness"].mean()),
}
profile


,dtype
spatial_tier,str
h3_cell,str
country_iso2,str
year,int64
month,int64
vector_group,str
pathogen_name,str
vector_present,int64
vector_detection_count,int64
individual_count_sum,int64


,spatial_tier,h3_cell,country_iso2,year,month,vector_group,pathogen_name,vector_present,vector_detection_count,individual_count_sum,...,pathogen_detection_count_within_10y,pathogen_ever_before,pathogen_ever_after,years_to_first_pathogen_after,years_since_last_pathogen_before,pathogen_same_month,pathogen_source_n_sequence,pathogen_source_n_outbreak,share_pathogen_exact_coords,share_pathogen_subnational_centroid
0,h3,8419519ffffffff,GB,1974,3,biting_midge,African horse sickness virus,1,14,11,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
1,h3,84195b7ffffffff,GB,1974,3,biting_midge,African horse sickness virus,1,14,38,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
2,h3,8419519ffffffff,GB,1974,4,biting_midge,African horse sickness virus,1,14,15,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
3,h3,84195b7ffffffff,GB,1974,4,biting_midge,African horse sickness virus,1,21,9,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN
4,h3,8419519ffffffff,GB,1974,5,biting_midge,African horse sickness virus,1,14,1,...,0,0,0,NaN,NaN,0,0,0,NaN,NaN


{'n_rows': 112134,
 'n_tier1_h3': 112134,
 'n_tier2_country': 0,
 'dup_tier1_keys': 0,
 'dup_tier2_keys': 0,
 'year_min': 1957,
 'year_max': 2025,
 'n_h3_cells': 2140,
 'n_countries': 57,
 'n_pathogens': 26,
 'vector_groups': ['biting_midge', 'mosquito', 'sandfly', 'tick'],
 'vector_present_all_one': True,
 'pathogen_positive_5y': 1740,
 'pathogen_positive_rate_5y': 0.015517149125153834,
 'pathogen_within_10y': 3417,
 'pathogen_ever_after': 6102,
 'pathogen_ever_before': 1388,
 'pathogen_same_month': 46,
 'species_richness_mean': 2.2825369647029445}

In [4]:
display(
    panel.groupby("pathogen_name", dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
        ever_after=("pathogen_ever_after", "sum"),
        ever_before=("pathogen_ever_before", "sum"),
        same_month=("pathogen_same_month", "sum"),
        mean_vector_detections=("vector_detection_count", "mean"),
    )
    .assign(positive_rate_5y=lambda d: d["within_5y"] / d["n_rows"])
    .sort_values("positive_rate_5y", ascending=False)
)

display(
    panel.groupby(["vector_group", "pathogen_name"], dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
    )
    .assign(positive_rate_5y=lambda d: d["within_5y"] / d["n_rows"])
)

display(
    panel.groupby("year", dropna=False)
    .agg(
        n_rows=("vector_present", "size"),
        within_5y=(TARGET_COL, "sum"),
        within_10y=("pathogen_within_10y", "sum"),
        ever_after=("pathogen_ever_after", "sum"),
    )
    .tail(20)
)


,n_rows,within_5y,within_10y,ever_after,ever_before,same_month,mean_vector_detections,positive_rate_5y
pathogen_name,,,,,,,,
Bluetongue virus,5038,426,1497,3050,436,4,9.651052,0.084557
West Nile virus,6674,552,1032,1489,300,14,7.206173,0.082709
Usutu virus,6674,499,532,574,481,24,7.206173,0.074768
Schmallenberg virus,5038,245,327,936,102,3,9.651052,0.048630
Sandfly fever Naples phlebovirus / Toscana virus,425,5,8,8,3,0,42.324706,0.011765
Tick-borne encephalitis virus,858,4,4,5,0,0,5.199301,0.004662
Alongshan virus,858,2,3,3,0,1,5.199301,0.002331
Dengue virus,6674,3,5,8,55,0,7.206173,0.000450
Zika virus,6674,3,3,6,7,0,7.206173,0.000450


n_rows  \
vector_group pathogen_name                                               
biting_midge African horse sickness virus                         5038   
             Akabane orthobunyavirus                              5038   
             Bluetongue virus                                     5038   
             Epizootic hemorrhagic disease virus                  5038   
             Schmallenberg virus                                  5038   
mosquito     Batai orthobunyavirus                                6674   
             Chikungunya virus                                    6674   
             Dengue virus                                         6674   
             Inkoo orthobunyavirus                                6674   
             Japanese encephalitis virus                          6674   
             Rift Valley fever virus                              6674   
             Sindbis virus                                        6674   
             Tahyna / California encephalitis orthobunyavirus     6674   
             Usutu virus                                          6674   
             West Nile virus                                      6674   
             Yellow fever virus                                   6674   
             Zika virus                                           6674   
sandfly      Phlebovirus (Mediterranean sandfly fever complex)     425   
             Sandfly fever Naples phlebovirus / Toscana virus      425   
tick         Alongshan virus                                       858   
             Crimean-Congo hemorrhagic fever virus                 858   
             Jingmen tick virus                                    858   
             Louping ill virus                                     858   
             Omsk hemorrhagic fever virus                          858   
             Thogoto / Bourbon thogotovirus                        858   
             Tick-borne encephalitis virus                         858   

                                                                within_5y  \
vector_group pathogen_name                                                  
biting_midge African horse sickness virus                               0   
             Akabane orthobunyavirus                                    0   
             Bluetongue virus                                         426   
             Epizootic hemorrhagic disease virus                        0   
             Schmallenberg virus                                      245   
mosquito     Batai orthobunyavirus                                      0   
             Chikungunya virus                                          0   
             Dengue virus                                               3   
             Inkoo orthobunyavirus                                      0   
             Japanese encephalitis virus                                0   
             Rift Valley fever virus                                    0   
             Sindbis virus                                              1   
             Tahyna / California encephalitis orthobunyavirus           0   
             Usutu virus                                              499   
             West Nile virus                                          552   
             Yellow fever virus                                         0   
             Zika virus                                                 3   
sandfly      Phlebovirus (Mediterranean sandfly fever complex)          0   
             Sandfly fever Naples phlebovirus / Toscana virus           5   
tick         Alongshan virus                                            2   
             Crimean-Congo hemorrhagic fever virus                      0   
             Jingmen tick virus                                         0   
             Louping ill virus                                          0   
             Omsk hemorrhagic fever virus                         

,n_rows,within_5y,within_10y,ever_after
year,,,,
2006,1850,2,202,252
2007,1739,15,129,171
2008,1537,19,35,84
2009,1668,28,65,105
2010,3357,40,99,143
2011,3147,161,230,287
2012,3419,66,89,163
2013,4560,133,152,262
2014,4653,81,99,178


## Missing values

Identity, vector features, and binary labels should be complete on the supervised panel.

Expected nulls:
- `share_pathogen_*` — only defined when the 5y forward window has events of **this** pathogen
- `years_to_first_pathogen_after` / `years_since_last_pathogen_before` — null when that side has no event (`pathogen_ever_after` / `pathogen_ever_before` == 0)


In [5]:
missing = (
    panel.isna()
    .sum()
    .rename("n_missing")
    .to_frame()
    .assign(pct=lambda d: 100.0 * d["n_missing"] / len(panel))
    .sort_values("n_missing", ascending=False)
)
display(missing.head(20))

share_qa = {
    "share_null_when_pathogen_0": float(
        panel.loc[panel[TARGET_COL] == 0, SHARE_COLS[0]].isna().mean()
    ),
    "share_null_when_pathogen_1": float(
        panel.loc[panel[TARGET_COL] == 1, SHARE_COLS[0]].isna().mean()
    ),
    "delay_after_null_when_ever_after_0": bool(
        panel.loc[panel["pathogen_ever_after"] == 0, "years_to_first_pathogen_after"].isna().all()
    ),
    "delay_before_null_when_ever_before_0": bool(
        panel.loc[panel["pathogen_ever_before"] == 0, "years_since_last_pathogen_before"].isna().all()
    ),
}
share_qa


,n_missing,pct
years_since_last_pathogen_before,110746,98.762195
share_pathogen_subnational_centroid,110394,98.448285
share_pathogen_exact_coords,110394,98.448285
years_to_first_pathogen_after,106032,94.558296
month,0,0.000000
vector_group,0,0.000000
pathogen_name,0,0.000000
year,0,0.000000
spatial_tier,0,0.000000
h3_cell,0,0.000000


{'share_null_when_pathogen_0': 1.0,
 'share_null_when_pathogen_1': 0.0,
 'delay_after_null_when_ever_after_0': True,
 'delay_before_null_when_ever_before_0': True}

## Outliers (vector effort proxies)

Detection and individual counts are right-skewed. Summarize tails and prepare winsor caps for **`COUNT_COLS` only** (group effort proxies). Do **not** winsorize priority `vector_count_*` species features. Do not drop units — rare pathogen positives must stay.

In [6]:
outlier_summary = []
winsor_caps: dict[str, float] = {}
for col in COUNT_COLS:
    s = panel[col]
    cap = float(s.quantile(COUNT_WINSOR_Q))
    winsor_caps[col] = cap
    outlier_summary.append(
        {
            "column": col,
            "min": float(s.min()),
            "p50": float(s.median()),
            "p95": float(s.quantile(0.95)),
            f"p{int(COUNT_WINSOR_Q * 100)}": cap,
            "max": float(s.max()),
            "n_above_cap": int((s > cap).sum()),
            "n_pos_above_cap": int(
                ((panel[TARGET_COL] == 1) & (s > cap)).sum()
            ),
        }
    )

display(pd.DataFrame(outlier_summary))
winsor_caps

,column,min,p50,p95,p99,max,n_above_cap,n_pos_above_cap
0,vector_detection_count,1.0,3.000000,22.0,64.0,7438.0,1120,50
1,individual_count_sum,0.0,9.000000,1155.0,7974.0,642933.0,1117,32
2,individual_count_mean,0.0,2.666667,200.0,949.0,321466.5,1118,25


{'vector_detection_count': 64.0,
 'individual_count_sum': 7974.0,
 'individual_count_mean': 949.0}

## Geo / time filters and modelling prep

Applied rules:

1. **Identity completeness** — drop rows missing required key fields for their `spatial_tier` (including `pathogen_name`).
2. **Valid calendar month** — `month` in 1–12.
3. **VectorNet coverage** — keep `vector_present == 1` (already true for the merge output).
4. **Modelling year window** — `year >= YEAR_MIN` (and `<= YEAR_MAX` if set).
5. **Feature prep** — winsorize vector effort `COUNT_COLS` at p99 only (not species `vector_count_*`). Forward-window `share_pathogen_*` stay nullable QA (not in `FEATURE_COLS`). Delay columns stay nullable and are not modelling features here.
6. **Species features (locked unit-grain pivot)** — use loaded `species_long` → distinct unit keys from filtered `panel_prep` → pivot **only** `priority_vector_taxa.csv` to `vector_count_<slug>` (0-fill) → left-merge onto pathogen-long rows → then set `PRIORITY_COUNT_COLS` / `FEATURE_COLS`. Row count must not change.

Row drops are reported; primary-label (`pathogen_within_5y`) retention is checked. Stratify later EDA by `pathogen_name`.


In [7]:
def apply_panel_prep(
    df: pd.DataFrame,
    *,
    year_min: int = YEAR_MIN,
    year_max: int | None = YEAR_MAX,
    winsor_q: float = COUNT_WINSOR_Q,
) -> tuple[pd.DataFrame, dict]:
    """Filter and prepare the modelling panel; return (prepared_df, filter_report)."""
    work = df.copy()
    n0 = len(work)
    n_pos0 = int(work[TARGET_COL].sum())
    report: dict = {"n_input": n0, "n_pos_input": n_pos0}

    # 1. Identity by tier
    h3_ok = (work["spatial_tier"] == "h3") & work["h3_cell"].notna()
    country_ok = (work["spatial_tier"] == "country") & work["country_iso2"].notna()
    common_ok = (
        work["spatial_tier"].isin(["h3", "country"])
        & work["year"].notna()
        & work["month"].notna()
        & work["vector_group"].fillna("").ne("")
        & work["pathogen_name"].fillna("").ne("")
        & work["country_iso2"].notna()  # metadata preferred on Tier 1 too
    )
    mask_id = common_ok & (h3_ok | country_ok)
    report["dropped_identity"] = int((~mask_id).sum())
    work = work.loc[mask_id].copy()

    # 2. Valid month
    mask_month = work["month"].between(1, 12)
    report["dropped_bad_month"] = int((~mask_month).sum())
    work = work.loc[mask_month].copy()

    # 3. VectorNet coverage
    mask_vec = work["vector_present"] == 1
    report["dropped_no_vector"] = int((~mask_vec).sum())
    work = work.loc[mask_vec].copy()

    # 4. Year window
    mask_year = work["year"] >= year_min
    if year_max is not None:
        mask_year &= work["year"] <= year_max
    report["dropped_year_window"] = int((~mask_year).sum())
    report["year_min"] = year_min
    report["year_max"] = year_max
    work = work.loc[mask_year].copy()

    # 5. Feature prep — winsorize group effort COUNT_COLS only (not species counts).
    caps = {c: float(work[c].quantile(winsor_q)) for c in COUNT_COLS}
    report["winsor_caps"] = caps
    report["winsor_q"] = winsor_q
    report["winsorize_cols"] = list(COUNT_COLS)
    for col, cap in caps.items():
        work[f"{col}_raw"] = work[col]
        work[col] = work[col].clip(upper=cap)

    work["year"] = work["year"].astype(int)
    work["month"] = work["month"].astype(int)

    report["n_output"] = len(work)
    report["n_pos_output"] = int(work[TARGET_COL].sum())
    report["n_dropped_total"] = n0 - len(work)
    report["pos_retained"] = report["n_pos_output"] == n_pos0 or report[
        "n_pos_output"
    ] == int(df.loc[df["year"] >= year_min, TARGET_COL].sum())
    return work.reset_index(drop=True), report


panel_prep, filter_report = apply_panel_prep(panel)

# 6. Species: locked closed priority set via unit-grain pivot, then merge
# (species_long already loaded; never join long×long onto pathogen-long).
n_prep_before_species = len(panel_prep)
panel_prep = attach_priority_species_features(panel_prep, species_long, _priority)
assert len(panel_prep) == n_prep_before_species

# Build modelling feature list only after pivot+merge (not from raw panel columns).
PRIORITY_COUNT_COLS = priority_count_columns(_priority)
FEATURE_COLS = [
    "vector_detection_count",
    "individual_count_sum",
    "individual_count_mean",
    "species_richness",
    *PRIORITY_COUNT_COLS,
    "pathogen_ever_before",
]
missing_priority = [c for c in PRIORITY_COUNT_COLS if c not in panel_prep.columns]
assert not missing_priority, missing_priority
assert PRIORITY_COUNT_COLS == [
    c for c in panel_prep.columns if c.startswith("vector_count_")
], "modelling vector_count_* must match locked priority set exactly"

filter_report["species_source"] = "modelling_panel_species.parquet"
filter_report["species_long_rows"] = int(len(species_long))
filter_report["n_priority_features"] = len(PRIORITY_COUNT_COLS)
filter_report["n_units_prep"] = int(
    panel_prep.loc[:, UNIT_KEY_COLS].drop_duplicates().shape[0]
)
filter_report


{'n_input': 112134,
 'n_pos_input': 1740,
 'dropped_identity': 0,
 'dropped_bad_month': 0,
 'dropped_no_vector': 0,
 'dropped_year_window': 29686,
 'year_min': 2010,
 'year_max': None,
 'winsor_caps': {'vector_detection_count': 70.0,
  'individual_count_sum': 9171.56999999992,
  'individual_count_mean': 1072.0},
 'winsor_q': 0.99,
 'winsorize_cols': ['vector_detection_count',
  'individual_count_sum',
  'individual_count_mean'],
 'n_output': 82448,
 'n_pos_output': 1675,
 'n_dropped_total': 29686,
 'pos_retained': True,
 'species_source': 'modelling_panel_species.parquet',
 'species_long_rows': 31108,
 'n_priority_features': 52,
 'n_units_prep': 9026}

In [8]:
prep_check_cols = IDENTITY_COLS + FEATURE_COLS + [
    TARGET_COL,
    "pathogen_within_10y",
    "pathogen_detection_count_within_5y",
    "pathogen_detection_count_within_10y",
    "pathogen_same_month",
]
prep_missing = panel_prep[prep_check_cols].isna().sum()
assert int(prep_missing.sum()) == 0, prep_missing[prep_missing > 0].to_dict()
assert "virus_within_5y" not in panel_prep.columns
assert "virus_detected" not in panel_prep.columns

{
    "prep_rows": len(panel_prep),
    "prep_pos_5y": int(panel_prep[TARGET_COL].sum()),
    "prep_pos_rate_5y": float(panel_prep[TARGET_COL].mean()),
    "prep_within_10y": int(panel_prep["pathogen_within_10y"].sum()),
    "prep_ever_after": int(panel_prep["pathogen_ever_after"].sum()),
    "prep_n_pathogens": int(panel_prep["pathogen_name"].nunique()),
    "prep_year_range": [int(panel_prep["year"].min()), int(panel_prep["year"].max())],
    "prep_missing_feature_cells": int(prep_missing.sum()),
    "spatial_tier_counts": panel_prep["spatial_tier"].value_counts().to_dict(),
}


{'prep_rows': 82448,
 'prep_pos_5y': 1675,
 'prep_pos_rate_5y': 0.020315835435668544,
 'prep_within_10y': 2447,
 'prep_ever_after': 2913,
 'prep_n_pathogens': 26,
 'prep_year_range': [2010, 2025],
 'prep_missing_feature_cells': 0,
 'spatial_tier_counts': {'h3': 82448}}

## Modelling handoff

`panel_prep` is the cleaned analysis table for later EDA / training cells / `train.py`.

| Object | Role |
|--------|------|
| `panel` | Raw merge output (pathogen-long; no merge-time `vector_count_*`) |
| `species_long` | Long composition from `modelling_panel_species.parquet` |
| `panel_prep` | Filtered + effort-winsorized + unit-pivoted priority species counts |
| `TARGET_COL` | `pathogen_within_5y` |
| `LABEL_COLS` | Forward / ever / delay / same-month label family (pathogen-specific) |
| `FEATURE_COLS` | Built **after** species pivot: effort + `species_richness` + locked `PRIORITY_COUNT_COLS` + `pathogen_ever_before` |
| `COUNT_COLS` | Only columns winsorized (group effort) — not `vector_count_*` |
| `filter_report` | Drop counts, winsor caps, species pivot metadata |

**Species side-table grain:** observed VectorNet **unit** × distinct `scientific_name` (**before** pathogen explosion). Schema is exactly `SPECIES_LONG_COLS` — **no** `pathogen_name`.

**Species pivot rule (locked for `train.py`):**
1. Filter panel (`apply_panel_prep`) — winsorize `COUNT_COLS` only.
2. Distinct `UNIT_KEY_COLS` from `panel_prep`.
3. Restrict `species_long` to those units.
4. Pivot closed `priority_vector_taxa.csv` → `vector_count_<feature_slug>` (absent → 0).
5. Left-merge wide counts onto pathogen-long rows (broadcast; row count unchanged).
6. Set `PRIORITY_COUNT_COLS` / `FEATURE_COLS` from that closed set.

Do not join species-long onto pathogen-long before pivoting. Do not winsorize priority species columns. Do not swap in frequency-selected taxa for the modelling handoff.

Use `pathogen_within_10y` as an alternate target for sensitivity runs. Stratify metrics by `pathogen_name`. Delay columns are for EDA histograms (`.dropna()`), not the primary supervised target.

Next (still Planned on the roadmap): feature distributions, target class-balance analysis (per pathogen), feature importance, spatial/temporal plots, then multi-model training.


In [9]:
X = panel_prep[FEATURE_COLS].copy()
y = panel_prep[TARGET_COL].astype(int).copy()
meta = panel_prep[
    IDENTITY_COLS
    + [
        "vector_present",
        "first_vector_year",
        "first_vector_month",
        "pathogen_detection_count_within_5y",
        "pathogen_detection_count_within_10y",
        "pathogen_within_10y",
        "pathogen_ever_before",
        "pathogen_ever_after",
        "pathogen_same_month",
        "years_to_first_pathogen_after",
        "years_since_last_pathogen_before",
    ]
].copy()

handoff = {
    "X_shape": X.shape,
    "y_shape": y.shape,
    "y_positive": int(y.sum()),
    "y_positive_rate": float(y.mean()),
    "n_pathogens": int(meta["pathogen_name"].nunique()),
    "feature_cols": FEATURE_COLS,
    "priority_count_cols": PRIORITY_COUNT_COLS,
    "winsorize_cols": COUNT_COLS,
    "target_col": TARGET_COL,
    "label_cols": LABEL_COLS,
    "species_path": str(SPECIES_PATH.relative_to(REPO_ROOT)),
    "filter_report": filter_report,
}
display(X[COUNT_COLS + ["species_richness", "pathogen_ever_before"]].describe().T)
handoff


,count,mean,std,min,25%,50%,75%,max
vector_detection_count,82448.0,6.135285,9.932493,1.0,1.0,3.0,6.0,70.00
individual_count_sum,82448.0,307.170559,1164.575385,0.0,2.0,13.0,99.0,9171.57
individual_count_mean,82448.0,47.426595,145.478777,0.0,1.0,4.0,24.5,1072.00
species_richness,82448.0,2.374042,2.248582,1.0,1.0,1.0,3.0,22.00
pathogen_ever_before,82448.0,0.016774,0.128425,0.0,0.0,0.0,0.0,1.00


{'X_shape': (82448, 57),
 'y_shape': (82448,),
 'y_positive': 1675,
 'y_positive_rate': 0.020315835435668544,
 'n_pathogens': 26,
 'feature_cols': ['vector_detection_count',
  'individual_count_sum',
  'individual_count_mean',
  'species_richness',
  'vector_count_culicoides',
  'vector_count_culicoides_achrayi',
  'vector_count_culicoides_chiopterus',
  'vector_count_culicoides_circumscriptus',
  'vector_count_culicoides_dewulfi',
  'vector_count_culicoides_festivipennis',
  'vector_count_culicoides_furcillatus',
  'vector_count_culicoides_imicola',
  'vector_count_culicoides_kingi',
  'vector_count_culicoides_lupicaris',
  'vector_count_culicoides_pallidicornis',
  'vector_count_culicoides_paolae',
  'vector_count_culicoides_puncticollis',
  'vector_count_culicoides_riethi',
  'vector_count_culicoides_scoticus',
  'vector_count_aedes',
  'vector_count_aedes_albopictus',
  'vector_count_aedes_caspius',
  'vector_count_aedes_japonicus',
  'vector_count_aedes_vexans_vexans',
  'vector_c